# LightGBM Pipeline A — 24 independent product models

Notebook này chỉ điều phối Pipeline A v2. Toàn bộ logic dữ liệu, feature engineering persona/history, train, lineage và submission nằm trong `src/` và được gọi qua entry points ở `scripts/`.

# 1. Bootstrap

Chạy cell bootstrap một lần để thiết lập source, secrets, dependencies và `SANTANDER_DATA_ROOT`. Không chạy lại nếu runtime hiện tại đã bootstrap thành công.

In [6]:
# ============================================================
# BOOTSTRAP: local + Colab CPU
#
# Local: d?ng tr?c ti?p source v? .env trong working tree.
# Colab browser: l?y config t? Colab Secrets.
# VS Code ? Colab CPU: d?n m?t COLAB_RUNTIME_CONFIG_B64 bundle duy nh?t.
# ============================================================
from __future__ import annotations

import base64
import binascii
import json
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

REPO_OWNER = "lbngyn"
REPO_NAME = "Santander-Product-Recommendation"
REPO_BRANCH = "feat/setup-pipeline"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
RUNTIME_CONFIG_KEYS = (
    "GITHUB_TOKEN",
    "GCP_SERVICE_ACCOUNT_JSON",
    "GOOGLE_CLOUD_PROJECT",
    "GCS_BUCKET",
    "GCS_RAW_PREFIX",
    "GCS_CHECKPOINT_PREFIX",
)


def is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Cannot find project root containing src/ and requirements.txt.")


def run_git(arguments: list[str], token: str) -> None:
    credentials = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    command = ["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {credentials}", *arguments]
    subprocess.run(command, check=True)


def decode_runtime_bundle(encoded_value: str) -> dict[str, str]:
    try:
        payload = base64.b64decode(encoded_value, validate=True).decode("utf-8")
        config = json.loads(payload)
    except (binascii.Error, UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("COLAB_RUNTIME_CONFIG_B64 must be a Base64-encoded JSON object.") from error
    missing = [key for key in RUNTIME_CONFIG_KEYS if not config.get(key)]
    if missing:
        raise ValueError(f"Runtime config bundle is missing: {', '.join(missing)}")
    if isinstance(config["GCP_SERVICE_ACCOUNT_JSON"], dict):
        config["GCP_SERVICE_ACCOUNT_JSON"] = json.dumps(config["GCP_SERVICE_ACCOUNT_JSON"])
    return {key: str(config[key]) for key in RUNTIME_CONFIG_KEYS}


def load_colab_ui_secrets() -> dict[str, str] | None:
    """Return secrets only when this kernel runs through the Colab browser UI."""
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
        if not github_token:
            return None
        config = {"GITHUB_TOKEN": github_token}
        for key in RUNTIME_CONFIG_KEYS[1:]:
            value = userdata.get(key)
            if not value:
                raise RuntimeError(f"Missing Colab Secret: {key}")
            config[key] = value
        return config
    except Exception:
        return None


def load_colab_runtime_config() -> dict[str, str]:
    """Use browser Secrets, env bundle, or one secure runtime prompt."""
    ui_config = load_colab_ui_secrets()
    if ui_config:
        print("Using Colab Secrets.")
        return ui_config

    encoded_value = os.getenv("COLAB_RUNTIME_CONFIG_B64")
    if not encoded_value:
        encoded_value = getpass("Paste COLAB_RUNTIME_CONFIG_B64 once (runtime-only): ")
    return decode_runtime_bundle(encoded_value)


IS_COLAB = is_colab_runtime()
ENV = "colab" if IS_COLAB else "local"
os.environ["SANTANDER_RUNTIME"] = ENV

if IS_COLAB:
    runtime_config = load_colab_runtime_config()
    os.environ.update(runtime_config)
    github_token = runtime_config["GITHUB_TOKEN"]
    PROJECT_ROOT = Path("/content") / REPO_NAME

    if not PROJECT_ROOT.exists():
        print("Cloning source to Colab fast disk...")
        run_git(["clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], github_token)
    else:
        print("Syncing latest source to Colab fast disk...")

        subprocess.run([
            "git", "-C", str(PROJECT_ROOT), "config",
            "remote.origin.fetch", "+refs/heads/*:refs/remotes/origin/*"
        ], check=True)

        run_git(["-C", str(PROJECT_ROOT), "fetch", "origin", "--prune"], github_token)
        run_git([
            "-C", str(PROJECT_ROOT),
            "checkout", "-B", REPO_BRANCH,
            f"origin/{REPO_BRANCH}"
        ], github_token)

    requirements_path = PROJECT_ROOT / "requirements.txt"
    if not requirements_path.is_file():
        raise FileNotFoundError(f"requirements.txt is missing from branch {REPO_BRANCH}.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)], check=True)

    # Persist raw CSV and Parquet checkpoints between Colab runtimes.
    from google.colab import drive
    drive_mount = Path("/content/drive")
    if not (drive_mount / "MyDrive").exists():
        drive.mount(str(drive_mount))
    DATA_ROOT = drive_mount / "MyDrive/projects/santander/data"
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    load_dotenv(PROJECT_ROOT / ".env")
    DATA_ROOT = Path(os.getenv("SANTANDER_DATA_ROOT", PROJECT_ROOT / "data"))

# DuckDB temporary spill files should stay on Colab's fast ephemeral disk.
if IS_COLAB:
    os.environ["SANTANDER_DUCKDB_TEMP_DIRECTORY"] = "/content/santander_duckdb_temp"

os.environ["SANTANDER_DATA_ROOT"] = str(DATA_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook ready | environment={ENV} | source={PROJECT_ROOT}")


Paste COLAB_RUNTIME_CONFIG_B64 once (runtime-only): ··········
Syncing latest source to Colab fast disk...
Notebook ready | environment=colab | source=/content/Santander-Product-Recommendation


In [11]:
# COLAB SOURCE UPDATE
# 1) Tr?n local: git add/commit/push source m?i l?n REPO_BRANCH.
# 2) Tr?n Colab: ch?y cell n?y, sau ?? ch?y l?i c?c cell import/EDA c?n d?ng code m?i.

if not IS_COLAB:
    print("Local environment: source is already current; no Colab sync is needed.")
else:
    status = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    if status:
        raise RuntimeError(
            "The Colab source copy has uncommitted changes. Do not overwrite it; "
            "restart the runtime or resolve those changes first."
        )

    run_git(["-C", str(PROJECT_ROOT), "fetch", "origin", REPO_BRANCH], github_token)
    run_git(["-C", str(PROJECT_ROOT), "checkout", REPO_BRANCH], github_token)
    run_git(["-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], github_token)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

    import importlib
    importlib.invalidate_caches()
    stale_modules = [name for name in sys.modules if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]
    print(f"Colab source updated from {REPO_BRANCH}; cleared {len(stale_modules)} cached src modules. Re-run the import/config cell, then the desired EDA cells.")


Colab source updated from feat/setup-pipeline; cleared 28 cached src modules. Re-run the import/config cell, then the desired EDA cells.


## 2. Train Pipeline A

Cấu hình versioned ở `configs/baselines/lightgbm_v2.yaml`. Lệnh dưới rebuild panel khi cấu hình yêu cầu, rồi tạo đủ 24 LightGBM acquisition models, model index và lineage manifest.

In [ ]:
# The notebook owns no pipeline logic; it calls the reusable entry point only.
from scripts.run_lightgbm_v2 import run_lightgbm_v2_from_config

result = run_lightgbm_v2_from_config()
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Create competition submission

The submission entry point reconstructs the same history features from the training history and June test profiles, scores all 24 artifacts, masks already-owned products, and writes the official Top-7 CSV.

In [15]:
from pathlib import Path
from scripts.run_lightgbm_v2 import run_lightgbm_v2_competition_from_config

submission = run_lightgbm_v2_competition_from_config(
    Path(result["artifacts"]) / "model_artifacts.json"
)
print({"submission": str(submission), "bytes": submission.stat().st_size})
submission

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Submission 1/24] scoring ind_ahor_fin_ult1
[Submission 2/24] scoring ind_aval_fin_ult1
[Submission 3/24] scoring ind_cco_fin_ult1
[Submission 4/24] scoring ind_cder_fin_ult1
[Submission 5/24] scoring ind_cno_fin_ult1
[Submission 6/24] scoring ind_ctju_fin_ult1
[Submission 7/24] scoring ind_ctma_fin_ult1
[Submission 8/24] scoring ind_ctop_fin_ult1
[Submission 9/24] scoring ind_ctpp_fin_ult1
[Submission 10/24] scoring ind_deco_fin_ult1
[Submission 11/24] scoring ind_dela_fin_ult1
[Submission 12/24] scoring ind_deme_fin_ult1
[Submission 13/24] scoring ind_ecue_fin_ult1
[Submission 14/24] scoring ind_fond_fin_ult1
[Submission 15/24] scoring ind_hip_fin_ult1
[Submission 16/24] scoring ind_nom_pens_ult1
[Submission 17/24] scoring ind_nomina_ult1
[Submission 18/24] scoring ind_plan_fin_ult1
[Submission 19/24] scoring ind_pres_fin_ult1
[Submission 20/24] scoring ind_reca_fin_ult1
[Submission 21/24] scoring ind_recibo_ult1
[Submission 22/24] scoring ind_tjcr_fin_ult1
[Submission 23/24] scoring

PosixPath('/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v2-history-persona-20260922T060901Z-21f6e4f4/submission.csv')